> **출처 고지**: 전력망 자료와 그 파생값(표·그림)은 © OpenStreetMap contributors, [ODbL 1.0](https://opendatacommons.org/licenses/odbl/1-0/)이다. 이 노트북의 결과물은 비공개이며, 공개하기 전에 보안 검토가 필요하다(PLAN 6.8, ADR 0014).

# 05_power_grid — 전력망 탐색 실험 (ADR 0014 제안, PLAN 6.8 일부를 앞당김)

7단계(다층 네트워크, v1 이후 백로그)의 작은 탐색만 이 세션에서 앞당긴다. v1 범위는 바꾸지 않는다. **탐색적이며 사전 판정 규칙이 없다.** 합격·불합격을 정하지 않고, 자료가 얼마나 차 있고 그래프가 어떤 모양인지만 기록한다.

## 규칙
- 자료는 OSM `power` 태그(Geofabrik south-korea-261017, ODbL)만 쓴다. 수급·계통 운영 자료와 DC 조류는 하지 않는다(PLAN 6.8: 이후 단계).
- 결과(표·그림·그래프 파일)는 저장소 밖 `C:\dev\kfm-data\power\`에만 쓴다. 저장소에는 `power_lib.py`와 출력을 지운 이 노트북만 둔다.
- 개별 선로·변전소의 취약점 목록은 만들지도 인용하지도 않는다. 이 노트북은 집계 수치만 출력한다.
- 0S형 탐색 코드라서 `kfm` 패키지·`runs`·스키마 규칙 밖이다(PLAN 6.1a). 대신 마지막에 `power_run_meta.json`으로 설정 해시, 자료 스냅숏, 코드 버전, 시드를 남긴다.
- 0S의 교훈: 곡률 해석은 엣지 길이의 의미에 달려 있다. 여기서는 곡률을 계산하지 않고, 엣지 가중치는 리액턴스 **대용값**(길이 / 전압²에 비례)이다. 후속에서 곡률을 쓰려면 엣지 길이의 의미(km, 리액턴스, hop)를 먼저 고정한다.

## 정의 (`power_lib.py`의 `CONFIG_DEFAULT`와 같다)
- **선로**: `power=line`·`cable` 중 전압 등급이 765·345·154 kV급(전압 태그 최댓값이 각각 600 kV 이상, 300–600, 100–300)인 것. `line=bay`·`busbar`(변전소 내부 모선·베이), `minor_line`(배전), 전압 태그가 없는 선로는 뺀다. HVDC(`frequency=0`)는 AC 그래프에서 빼고 민감도에서만 넣는다.
- **노드**: 변전소 부지와 선로 끝점. 변전소는 `power=substation`(노드·방향·관계) 중 전압이 100 kV 이상이거나 미상인 것이다. 선로 끝점이 변전소 도형에서 300 m 안이면 그 변전소에 붙인다(전압이 확인된 변전소 우선, 같으면 가까운 쪽). 맞닿거나 겹친 변전소 도형은 한 부지로 합친다.
- **엣지**: 선로 구간. way 양 끝, 선로끼리 공유한 꼭짓점, 변전소 도형 안의 내부 꼭짓점에서 나눈다. 양 끝이 같은 노드인 구간은 뺀다.
- **리액턴스 대용값**: x = 0.35 Ω/km · L · 100 MVA / V² (p.u.). 가공선 일반값을 전압과 상관없이 쓴 대용값이며 변압기 임피던스는 무시한다(변전소 노드 하나가 여러 전압을 잇는다). 실제 선로 정수가 아니다.
- **유효 저항**: 전도도 1/x인 저항 회로망의 R_eff(u, v) = (e_u − e_v)ᵀ L⁺ (e_u − e_v). 노드 값 `rbar`는 같은 연결 성분의 다른 노드까지 R_eff의 평균이다(집계는 최대 성분 노드만). 엣지 레버리지 = R_eff(e) / x_e이고, 1이면 브리지이며 전체 합은 n − (성분 수)다(Foster 정리).
- **N-1 (엣지 제거)**: 구간 단위는 OSM 구간 하나(회랑 전체)를 끊을 때 성분이 나뉘는 엣지(브리지)다. 회선 단위는 회선 수(`circuits`, 없으면 `cables` / 3, 모르면 1)만큼 병렬 엣지로 펼친 뒤의 브리지다.

In [ ]:
# 05_power_grid 설정 셀: 경로, 설정값, 자료 MD5 점검, 출력 폴더, 저장 보조 함수
import csv
import datetime
import hashlib
import importlib.metadata as md
import json
import platform
import subprocess
import sys
import time
from pathlib import Path

sys.dont_write_bytecode = True  # 저장소에 __pycache__를 만들지 않는다

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from scipy import stats

import power_lib as pl

T_START = time.time()
RAW = Path(r"C:\dev\kfm-data\raw")
DATA = Path(r"C:\dev\kfm-data\spike")
OUT = Path(r"C:\dev\kfm-data\power")  # 모든 결과(표·그림·그래프)는 여기에만 쓴다
PBF_REL = "geofabrik_20261018/south-korea-261017.osm.pbf"
PBF = RAW / PBF_REL
NB_DIR = Path.cwd()  # notebooks/spike/power (nbconvert는 노트북 폴더에서 실행한다)
assert (NB_DIR / "power_lib.py").exists(), "notebooks/spike/power 폴더에서 실행해야 한다"
assert "kfm-spike" in sys.executable, "커널이 kfm-spike 환경이 아니다"
REPO = NB_DIR.parents[2]
assert (REPO / "CLAUDE.md").exists(), "저장소 위치를 찾지 못했다"
assert REPO.resolve() not in [OUT.resolve(), *OUT.resolve().parents], "결과 폴더가 저장소 안에 있다"
assert "onedrive" not in str(OUT).lower(), "결과 폴더가 OneDrive 안에 있다"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "00_README_비공개.txt").write_text(
    "전력망 탐색 실험(ADR 0014)의 결과물이다. 비공개이며 공개 전에 보안 검토가 필요하다(PLAN 6.8).\n"
    "저장소, OneDrive, 클라우드에 올리지 않는다. 개별 선로·변전소 단위 내용은 인용하지 않는다.\n"
    "자료: (c) OpenStreetMap contributors, ODbL 1.0 (파생물을 공개할 때 고지한다).\n",
    encoding="utf-8",
)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
SEED = 0  # 합성 검산의 무작위 그래프에만 쓴다. 본 계산에는 무작위가 없다

# 설정값(결과를 바꾸는 값). 기준은 pl.CONFIG_DEFAULT이고, 아래 시나리오에서만 바꾼다
CFG = dict(pl.CONFIG_DEFAULT)
SCENARIOS = {
    "기준": {},
    "부착 반경 100 m": {"attach_radius_m": 100.0},
    "부착 반경 500 m": {"attach_radius_m": 500.0},
    "끝점 병합 100 m": {"snap_m": 100.0},
    "변전소 부지 병합 300 m": {"site_merge_m": 300.0},
    "HVDC 포함": {"include_hvdc": True},
    "345 kV 이상만": {"grades": ("765", "345")},
    "변전소 안 꼭짓점 분할 끔": {"split_in_sites": False},
}
CONTRACT_NAME = "직렬 노드 합침(기준 그래프에서)"
CONFIG_HASH = hashlib.sha256(
    json.dumps({"base": CFG, "scenarios": SCENARIOS, "contract": CONTRACT_NAME}, sort_keys=True, default=str).encode()
).hexdigest()[:16]


def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


manifest = {r["path"]: r for r in csv.DictReader(open(RAW / "manifest.csv", encoding="utf-8-sig"))}
PBF_MD5 = md5_of(PBF)
assert PBF_MD5 == manifest[PBF_REL]["md5"], "PBF의 MD5가 manifest와 다르다"

WRITTEN = []  # 이 실행이 OUT에 쓴 파일


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return None if np.isnan(o) else float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def save_csv(df, name):
    df.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    WRITTEN.append(name)


def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return float(o) if np.isfinite(o) else None
    return o


def save_json(obj, name):
    (OUT / name).write_text(json.dumps(_clean(obj), ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    WRITTEN.append(name)


def save_parquet(df, name):
    df.to_parquet(OUT / name, index=False)
    WRITTEN.append(name)


def save_fig(fig, name):
    fig.savefig(OUT / name, dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
    WRITTEN.append(name)
    plt.close(fig)


def pf(df, digits=3):
    return df.to_string(index=False, float_format=lambda x: f"{x:,.{digits}f}")


print(f"설정 해시 {CONFIG_HASH}, PBF MD5 {PBF_MD5} (manifest와 일치)")
print("환경:", platform.python_version(), {m: md.version(m) for m in ("duckdb", "geopandas", "shapely", "numpy", "pandas", "scipy", "pyproj")})
print("결과 폴더:", OUT)

## 1. 합성 검산 (실행 결과를 남긴다)

실데이터 앞에서 `power_lib.py`의 함수를 합성 그래프로 먼저 검산한다(CLAUDE.md: 새 지표는 합성 그래프로 먼저 테스트). 경로·사이클·완전그래프·barbell·트리의 브리지와 유효 저항 해석해, 무작위 다중 그래프 300개에 대한 브리지(엣지 하나씩 지워 본 결과와 비교)·Foster 정리·유사역행렬 직접 계산, 장난감 지도의 그래프 조립(부착·분할·자기 루프·병합·HVDC)을 본다. 하나라도 어긋나면 `AssertionError`로 멈춘다.

In [ ]:
# 1-1. 합성 검산
synth = pl.run_synthetic_checks(seed=SEED)
assert (synth["결과"] == "OK").all()
save_csv(synth, "synthetic_checks.csv")
print(f"합성 검산 {len(synth)}항목 모두 통과 (오차 0이 아닌 항목의 최대 오차 {synth['오차'].max():.2e})")
print("항목 예:", " | ".join(synth["항목"].iloc[[0, 12, 20, 30, 40, len(synth) - 1]]))

## 2. 자료 읽기와 목록

DuckDB `ST_ReadOSM`으로 PBF에서 `power=line`·`minor_line`·`cable`(voltage, circuits, cables, frequency, location), `power=substation`(voltage, substation), `power=plant`·`generator`(출력·원료), `power=converter`를 읽는다. 전압은 `;`로 나뉜 다중 값을 모두 읽어 최댓값으로 등급을 매긴다. 채움률은 "태그가 있는 비율"과 "해석 가능한 비율"을 따로 센다.

In [ ]:
# 2-1. 읽기와 분류
con = pl.connect_osm()
osm = pl.read_power_osm(con, PBF)
lines = pl.classify_lines(osm["lines"])
subs = pl.classify_subs(osm["subs"])
print(
    f"읽기 완료({time.time() - T_START:.0f}s): 선로 way {len(lines):,}개(line·minor_line·cable), "
    f"변전소 {len(subs):,}개(노드 {(subs.kind == 'node').sum()}, way {(subs.kind == 'way').sum()}, 관계 {(subs.kind == 'relation').sum()}), "
    f"발전소 {len(osm['plants']):,}개, 발전기 {len(osm['generators']):,}개, 변환소 {len(osm['converters'])}개"
)
print("선로 도형 좌표가 2점 미만인 way:", int((lines["xy"].map(len) < 2).sum()), "| 변전소 도형이 비어 있는 것:", int(subs.geometry.is_empty.sum()))

In [ ]:
# 2-2. 자료 목록: 선로, 변전소, 발전소·발전기 (집계만)
inv_l, inv_s = pl.inventory_lines(lines), pl.inventory_subs(subs)
inv_p, inv_g = pl.inventory_units(osm["plants"]), pl.inventory_units(osm["generators"])
for k, df in inv_l.items():
    save_csv(df, f"inventory_lines_{k}.csv")
for k, df in inv_s.items():
    save_csv(df, f"inventory_substations_{k}.csv")
for k, df in inv_p.items():
    save_csv(df, f"inventory_plants_{k}.csv")
for k, df in inv_g.items():
    save_csv(df, f"inventory_generators_{k}.csv")

print("[선로] 전압 등급 x 종류 (변전소 내부 모선·베이 제외). 길이는 EPSG:5179 폴리라인 길이(km)")
print(pf(inv_l["by_grade"]))
print("\n[선로] 태그 채움률 (종류별, 내부 모선·베이 제외)")
print(pf(inv_l["fill"]))
print("\n[선로] 변전소 내부 모선·베이(line=bay·busbar)")
print(pf(inv_l["internal"]))
print("\n[선로] 전압 태그 원문 값(3개 이상인 것만)과 배정 등급")
print(pf(inv_l["raw_voltage"].query("n_ways >= 3")))

In [ ]:
# 2-3. 변전소·발전소·발전기
print("[변전소] 종류별 태그 채움률"); print(pf(inv_s["by_kind"]))
print("\n[변전소] 전압 등급별 수"); print(pf(inv_s["by_grade"], 0))
print("\n[변전소] 154 kV 이상 중 substation= 유형"); print(pf(inv_s["by_type_hv"], 0))
print("\n[발전소 power=plant] 종류별 출력·원료 채움률(출력 숫자 채움률은 단위가 붙은 값만)"); print(pf(inv_p["by_kind"]))
print("\n[발전소] 원료별(첫 원료 기준)"); print(pf(inv_p["by_source"].head(14)))
print("\n[발전기 power=generator] 종류별"); print(pf(inv_g["by_kind"]))
print("\n[변환소 power=converter]", osm["converters"].groupby(["voltage", "frequency"]).size().rename("n").reset_index().to_dict("records"))

## 3. 송전 그래프 (154 kV 이상)

노드는 변전소 부지와 선로 끝점이다. 끝점은 300 m 안의 변전소에 붙이고, 엣지는 선로 구간이다. 연결 성분과 최대 성분 비율을 내고, 765 kV 골격이 하나로 이어지는지 본다.

In [ ]:
# 3-1. 그래프 조립과 지표
gb = pl.build_graph(lines, subs, CFG)
nodes0, edges0, DIAG = gb["nodes"], gb["edges"], gb["diag"]
A = pl.analyze_graph(nodes0, edges0)
nodes1, edges1, S = A["nodes"], A["edges"], A["summary"]
print("[조립 진단]"); print(pd.Series(DIAG).to_string())
keys = ["n_nodes", "n_nodes_substation", "n_nodes_junction", "n_nodes_end", "n_edges", "length_km_total", "n_components", "comp_sizes_top5",
        "n_components_size_le3", "lc_nodes", "lc_node_share", "lc_edges", "lc_length_share"]
print("\n[그래프 규모와 연결성]"); print(pd.Series({k: S[k] for k in keys}).to_string())
print(f"\n전압 등급별 엣지: {edges1.groupby('grade').size().to_dict()}, 길이(km): { {g: round(v, 1) for g, v in edges1.groupby('grade')['length_km'].sum().items()} }")
print("회선 수를 아는 엣지 비율: 개수 {:.1%}, 길이 {:.1%}".format(edges1["circuits_n"].notna().mean(), edges1.loc[edges1["circuits_n"].notna(), "length_km"].sum() / edges1["length_km"].sum()))
print("지중 케이블(power=cable) 길이 비율: {:.1%}".format(edges1.loc[edges1["power"] == "cable", "length_km"].sum() / edges1["length_km"].sum()))

In [ ]:
# 3-2. 전압 등급별 골격이 하나로 이어지는지 (765 kV 정성 확인용 집계)
SUB = {g: pl.subgraph_summary(nodes1, edges1, (edges1["grade"] == g).to_numpy()) for g in ("765", "345", "154")}
SUB["765+345"] = pl.subgraph_summary(nodes1, edges1, edges1["grade"].isin(["765", "345"]).to_numpy())
tab = pd.DataFrame(SUB).T[["n_edges", "n_nodes", "length_km", "n_components", "lc_length_share", "gap_km_between_top2", "share_nodes_shared_with_other_edges", "share_length_in_graph_lc"]].astype(float)
print(pf(tab.reset_index(names="등급")))
print("\n765 kV 조각별 길이(km, 내림차순):", SUB["765"]["comp_length_km_desc"], "| 345 kV:", SUB["345"]["comp_length_km_desc"][:5], "...")
n765 = SUB["765"]["n_components"]
print(
    f"정성 판단: 765 kV 구간은 OSM에서 {n765}개 조각으로 보인다. "
    + ("하나로 이어진다." if n765 == 1 else f"가장 큰 두 조각 사이 최단 노드 간 거리는 {SUB['765']['gap_km_between_top2']:.1f} km다. ")
    + f"765 kV 길이의 {SUB['765']['share_length_in_graph_lc']:.0%}가 전체 그래프의 최대 성분 안에 있어, 345·154 kV를 거치면 한 계통으로 이어진다."
)

In [ ]:
# 3-3. 그림 1: 전압 등급별 송전 그래프와 765 kV 조각 (비공개, 저장소 밖에만 저장)
INK, INK2, GRID_C, BG = "#0b0b0b", "#52514e", "#d9d8d3", "#fcfcfb"
BLUE_LO, BLUE_MID, BLUE_HI, ORANGE, GREEN, GRAY = "#6da7ec", "#2a78d6", "#104281", "#eb6834", "#1baf7a", "#b8b7b2"
FOOT = "비공개 · 공개 전 보안 검토 필요(ADR 0014) · © OpenStreetMap contributors (ODbL)"
regions = gpd.read_parquet(DATA / "regions.parquet")
assert regions.crs.to_epsg() == 5179 and len(regions) == 250
sido_poly = regions.dissolve(by="sido_code")
allxy = np.vstack([np.vstack(list(edges1["coords"])), *lines.loc[lines["hvdc"], "xy"]])
PAD = 15000.0  # 지도 범위는 그래프(와 HVDC 케이블)에 맞춘다. 독도·먼 섬은 넣지 않는다
EXT = (allxy[:, 0].min() - PAD, allxy[:, 0].max() + PAD, allxy[:, 1].min() - PAD, allxy[:, 1].max() + PAD)


def base_map(ax):
    sido_poly.boundary.plot(ax=ax, color=GRID_C, linewidth=0.6, zorder=1)
    ax.set_xlim(EXT[0], EXT[1])
    ax.set_ylim(EXT[2], EXT[3])
    ax.set_aspect("equal")
    ax.axis("off")


def segs_of(edges, mask):
    return list(edges.loc[mask, "coords"])


fig, axes = plt.subplots(1, 2, figsize=(13, 8.5), facecolor=BG)
ax = axes[0]
base_map(ax)
for g, col, lw in (("154", BLUE_LO, 0.7), ("345", BLUE_MID, 1.2), ("765", BLUE_HI, 2.4)):
    ax.add_collection(LineCollection(segs_of(edges1, edges1["grade"] == g), colors=col, linewidths=lw, zorder=3))
ax.add_collection(LineCollection(list(lines.loc[lines["hvdc"], "xy"]), colors=ORANGE, linewidths=1.3, linestyles="--", zorder=4))
sites = nodes1.loc[nodes1["is_site"]]
ax.scatter(sites["x"], sites["y"], s=3, c=INK2, linewidths=0, zorder=5)
ax.legend(
    handles=[
        Line2D([0], [0], color=BLUE_LO, lw=1.2, label="154 kV급"),
        Line2D([0], [0], color=BLUE_MID, lw=1.8, label="345 kV급"),
        Line2D([0], [0], color=BLUE_HI, lw=2.6, label="765 kV급"),
        Line2D([0], [0], color=ORANGE, lw=1.4, ls="--", label="HVDC 해저 케이블(AC 그래프에서 제외)"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor=INK2, markersize=3.5, label="변전소 노드"),
    ],
    loc="lower right", frameon=True, facecolor=BG, edgecolor="none", framealpha=0.92, fontsize=9, labelcolor=INK2,
)
ax.set_title("송전 그래프: 전압 등급별 (OSM, 154 kV 이상)", fontsize=11, color=INK, loc="left")

ax = axes[1]
base_map(ax)
ax.add_collection(LineCollection(segs_of(edges1, np.ones(len(edges1), bool)), colors="#c9c8c2", linewidths=0.6, zorder=2))
comp765 = pl.subgraph_edge_components(nodes1, edges1, (edges1["grade"] == "765").to_numpy())
handles = []
for k, length_km in enumerate(comp765["comp_length_km"]):
    col = [BLUE_MID, ORANGE, GREEN][k] if k < 3 else GRAY
    ax.add_collection(LineCollection(segs_of(edges1, comp765["edge_comp"] == k), colors=col, linewidths=2.8, zorder=4))
    handles.append(Line2D([0], [0], color=col, lw=2.8, label=f"765 kV 조각 {k + 1}: {length_km:,.0f} km"))
handles.append(Line2D([0], [0], color="#c9c8c2", lw=1.2, label="그 밖의 송전선(154·345 kV)"))
ax.legend(handles=handles, loc="lower right", frameon=True, facecolor=BG, edgecolor="none", framealpha=0.92, fontsize=9, labelcolor=INK2)
ax.set_title("765 kV 구간의 연결 조각 (765 kV 엣지만으로 본 연결 성분)", fontsize=11, color=INK, loc="left")
fig.text(0.01, 0.005, FOOT, fontsize=8, color=INK2)
save_fig(fig, "fig1_network_voltage_765.png")
print("그림 1 저장:", OUT / "fig1_network_voltage_765.png")

In [ ]:
# 3-4. 그래프 파일 (ADR 0012 스키마 모양으로): network_nodes, network_edges, network_edges_power, layer_coupling(located_in)
reg = pl.assign_regions(nodes1[["x", "y"]].to_numpy(), regions, CFG["nearest_region_max_km"])
nodes2 = pd.concat([nodes1.reset_index(drop=True), reg], axis=1)
nodes2.insert(0, "node_id", np.arange(len(nodes2)))
print("노드의 시군구 귀속(located_in):", reg["located_in_method"].value_counts().to_dict(), "| 가장 먼 nearest 거리 {:.0f} m".format(float(reg["dist_m"].max())))

SRC = "osm_geofabrik_20261018"
g_nodes = gpd.GeoDataFrame(
    {"node_id": nodes2["node_id"], "layer": "power", "source_id": SRC + ":" + nodes2["osm_ref"].astype(str)},
    geometry=gpd.points_from_xy(nodes2["x"], nodes2["y"]), crs=pl.CRS_WORK,
)
g_edges = gpd.GeoDataFrame(
    {"edge_id": edges1["edge_id"], "layer": "power", "from_node": edges1["u"], "to_node": edges1["v"], "directed": False,
     "length_km": edges1["length_km"], "source_id": SRC + ":way/" + edges1["way_id"].astype(str)},
    geometry=[shapely.LineString(c) for c in edges1["coords"]], crs=pl.CRS_WORK,
)
ext = pd.DataFrame(
    {"edge_id": edges1["edge_id"], "voltage_kv": edges1["kv"], "reactance": edges1["x_pu"], "capacity_mw": np.nan,
     "circuits": edges1["circuits_n"], "grade": edges1["grade"], "power": edges1["power"], "underground": edges1["underground"]}
)
coupling = pd.DataFrame(
    {"coupling_id": np.arange(len(nodes2)), "src_layer": "power", "src_node_id": nodes2["node_id"], "dst_kind": "region",
     "dst_layer": pd.Series([None] * len(nodes2), dtype="string"), "dst_id": nodes2["unit_code"], "relation": "located_in", "weight": 1.0,
     "valid_from": pd.Series([None] * len(nodes2), dtype="string"), "valid_to": pd.Series([None] * len(nodes2), dtype="string"),
     "source_id": "spike_regions_2025-12-31"}
)
node_metrics = nodes2.copy()
edge_metrics = edges1.drop(columns=["coords"])
g_nodes.to_parquet(OUT / "network_nodes.parquet"); WRITTEN.append("network_nodes.parquet")
g_edges.to_parquet(OUT / "network_edges.parquet"); WRITTEN.append("network_edges.parquet")
save_parquet(ext, "network_edges_power.parquet")
save_parquet(coupling, "layer_coupling_located_in.parquet")
save_parquet(node_metrics, "power_node_metrics.parquet")
save_parquet(edge_metrics, "power_edge_metrics.parquet")
print("ADR 0012 모양 표: network_nodes", g_nodes.shape, "| network_edges", g_edges.shape, "| network_edges_power", ext.shape, "| layer_coupling(located_in)", coupling.shape)

## 4. 탐색 지표

리액턴스 대용값으로 유효 저항을 구하고, N-1 엣지 제거(브리지)를 센 뒤, 노드를 시군구(`located_in`)에 귀속해 시도별로 모은다. DC 조류와 수급 자료는 하지 않는다.

- 브리지 비율은 엣지 기준이다. 말단 가지(한쪽이 노드 하나뿐인 브리지)가 많으면 부풀려지므로 "말단이 아닌 브리지"를 따로 센다.
- 회선 단위 N-1은 `circuits`를 모르는 구간을 1회선으로 보므로 **상한**이다.
- 시도 집계의 `rbar`는 최대 성분 안 노드만으로 낸다. 최대 성분 밖(제주 AC 계통 등)은 `rbar_mean_own_comp`에만 값이 있고 서로 비교할 수 없다.

In [ ]:
# 4-1. 전국 요약 지표 (기준 그래프)
M_KEYS = ["n_bridges", "bridge_share", "bridge_length_share", "n_bridges_lc", "bridge_share_lc", "n_pendant_edges", "n_bridges_nonpendant",
        "n_bridges_circuit_level", "bridge_share_circuit_level", "nodes_touching_bridge_share", "nodes_touching_bridge_share_lc",
        "nodes_touching_nonpendant_bridge_share_lc", "bridge_small_side_median", "bridge_small_side_p90",
        "rbar_mean_lc", "rbar_median_lc", "kirchhoff_lc", "leverage_mean", "foster_residual"]
print(pd.Series({k: S[k] for k in M_KEYS}).to_string())
lev = edges1["leverage"].to_numpy()
SHARE_LEV90 = float((lev >= 0.9).mean())
print(f"레버리지 0.9 이상 엣지 비율 {SHARE_LEV90:.1%} (브리지 포함), 레버리지 0.5 미만 {float((lev < 0.5).mean()):.1%}")
assert S["foster_residual"] < 1e-6, "Foster 정리 잔차가 크다"

In [ ]:
# 4-2. 시도별 집계 (노드 수, 브리지에 닿는 노드 비율, 평균 유효 저항)
sido1 = pl.aggregate_by_sido(nodes2)
save_csv(sido1, "sido_summary.csv")
cols = ["sido_name", "n_nodes", "n_substation_nodes", "n_nodes_lc", "lc_share", "share_touch_bridge_lc", "share_touch_nonpendant_bridge_lc",
        "share_touch_bridge_circuit_lc", "rbar_mean_lc", "rbar_mean_own_comp", "lev_mean_lc"]
print(pf(sido1[cols]))

In [ ]:
# 4-3. 민감도: 부착 반경, 끝점 병합, 부지 병합, HVDC, 345 kV 이상, 직렬 노드 합침
SENS_KEYS = ["n_nodes", "n_edges", "length_km_total", "n_components", "lc_node_share", "lc_length_share", "n_bridges", "bridge_share",
             "n_bridges_nonpendant", "bridge_share_circuit_level", "nodes_touching_bridge_share_lc", "rbar_mean_lc"]
rows = []
for name, over in SCENARIOS.items():
    g = pl.build_graph(lines, subs, {**CFG, **over})
    s = pl.analyze_graph(g["nodes"], g["edges"])["summary"]
    rows.append({"scenario": name, **{k: s[k] for k in SENS_KEYS}, "comp_sizes_top3": "/".join(str(x) for x in s["comp_sizes_top5"][:3])})
ct = pl.contract_series_nodes(nodes0, edges0)
sc = pl.analyze_graph(ct["nodes"], ct["edges"])
rows.append({"scenario": CONTRACT_NAME, **{k: sc["summary"][k] for k in SENS_KEYS}, "comp_sizes_top3": "/".join(str(x) for x in sc["summary"]["comp_sizes_top5"][:3])})
sens = pd.DataFrame(rows)
save_csv(sens, "sensitivity.csv")
print(pf(sens))
print(f"\n직렬 노드 합침으로 사라진 노드: {ct['n_contracted']}개 (차수 2인 비변전소 노드)")
# 합친 그래프로 시도 집계를 다시 낸다
regc = pl.assign_regions(sc["nodes"][["x", "y"]].to_numpy(), regions, CFG["nearest_region_max_km"])
sido_c = pl.aggregate_by_sido(pd.concat([sc["nodes"].reset_index(drop=True), regc], axis=1))
save_csv(sido_c, "sido_summary_contracted.csv")
cmp_ = sido1[["sido_name", "n_nodes", "share_touch_bridge_lc", "rbar_mean_lc"]].merge(
    sido_c[["sido_name", "n_nodes", "share_touch_bridge_lc", "rbar_mean_lc"]], on="sido_name", suffixes=("", "_합침")
)
print(pf(cmp_))

In [ ]:
# 4-4. 가중치 민감도: 회선 수를 반영한 리액턴스(x / 회선 수)로 rbar를 다시 구해 순위 상관을 본다
e = edges1
circ = np.where(e["circuits_n"].to_numpy(float) >= 1, e["circuits_n"].to_numpy(float), 1.0)
er2 = pl.effective_resistance(len(nodes1), e["u"].to_numpy(int), e["v"].to_numpy(int), circ / e["x_pu"].to_numpy(float))
inlc = nodes1["in_lc"].to_numpy()
rho_node = float(stats.spearmanr(nodes1.loc[inlc, "rbar"], er2["rbar"][inlc]).statistic)
t = pd.DataFrame({"sido_name": nodes2["sido_name"], "r1": nodes1["rbar"], "r2": er2["rbar"], "in_lc": inlc})
t = t.loc[t["in_lc"]].groupby("sido_name").agg(n=("r1", "size"), r1=("r1", "mean"), r2=("r2", "mean"))
t = t.loc[t["n"] >= 10]
rho_sido = float(stats.spearmanr(t["r1"], t["r2"]).statistic)
RBAR_CIRC = {"rho_node": rho_node, "rho_sido": rho_sido, "n_sido": int(len(t)), "rbar_mean_lc_circuit_adjusted": float(np.nanmean(er2["rbar"][inlc]))}
print(f"회선 수 반영 rbar와 기준 rbar의 Spearman: 노드(최대 성분) {rho_node:.3f}, 시도 평균({len(t)}개 시도) {rho_sido:.3f}")

In [ ]:
# 4-5. 그림 2: 브리지 구조 지도와 시도별 비율 (비공개, 저장소 밖에만 저장)
RED = "#e34948"
cls = np.zeros(len(edges1), dtype=int)  # 0 복수 경로, 1 말단 가지 브리지, 2 말단이 아닌 브리지, 3 말단이 아닌 브리지이면서 회선 단위에서도 브리지
cls[edges1["is_bridge"].to_numpy()] = 1
cls[edges1["bridge_nonpendant"].to_numpy()] = 2
cls[(edges1["bridge_nonpendant"] & edges1["is_bridge_circuit"]).to_numpy()] = 3
style = {0: (GRAY, 0.7, 2), 1: (BLUE_LO, 1.0, 3), 2: (ORANGE, 1.9, 4), 3: (RED, 2.6, 5)}
fig, axes = plt.subplots(1, 2, figsize=(14, 8.5), facecolor=BG, gridspec_kw={"width_ratios": [1.0, 0.9]})
ax = axes[0]
base_map(ax)
labels = {0: "복수 경로(브리지 아님)", 1: "말단 가지 브리지", 2: "말단이 아닌 브리지(구간 단위)", 3: "말단이 아닌 브리지 + 회선 단위에서도 브리지"}
handles = []
for c, (col, lw, z) in style.items():
    ax.add_collection(LineCollection(segs_of(edges1, cls == c), colors=col, linewidths=lw, zorder=z))
    handles.append(Line2D([0], [0], color=col, lw=max(lw, 1.4), label=f"{labels[c]}: {int((cls == c).sum())}개"))
ax.legend(handles=handles, loc="lower right", bbox_to_anchor=(1.0, 0.08), frameon=True, facecolor=BG, edgecolor="none", framealpha=0.92, fontsize=8.5, labelcolor=INK2)
ax.set_title("구조적 N-1: 브리지 구간 (기준 그래프)", fontsize=11, color=INK, loc="left")

ax = axes[1]
d = sido1.loc[(sido1["sido_name"] != "전국") & (sido1["n_nodes_lc"] >= 10)].sort_values("n_nodes_lc")
ypos = np.arange(len(d))
ax.barh(ypos + 0.2, d["share_touch_bridge_lc"] * 100, height=0.36, color=BLUE_MID, label="구간 단위 브리지에 닿는 노드(최대 성분 안)")
ax.barh(ypos - 0.2, d["share_touch_bridge_circuit_lc"] * 100, height=0.36, color=ORANGE, label="회선 단위 브리지에 닿는 노드(상한)")
ax.set_yticks(ypos)
ax.set_yticklabels([f"{n} ({k})" for n, k in zip(d["sido_name"], d["n_nodes_lc"], strict=True)], fontsize=9, color=INK2)
ax.set_xlabel("노드 비율 (%)", color=INK2)
ax.set_xlim(0, 105)
ax.set_xticks(range(0, 101, 20))
ax.grid(axis="x", color=GRID_C, linewidth=0.6)
ax.set_axisbelow(True)
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.tick_params(colors=INK2, length=0)
ax.legend(loc="upper center", bbox_to_anchor=(0.4, -0.1), frameon=False, fontsize=9, labelcolor=INK2)
ax.set_title("시도별 브리지에 닿는 노드 비율 (괄호: 최대 성분 노드 수, 10개 이상만)", fontsize=11, color=INK, loc="left")
fig.text(0.01, 0.005, FOOT, fontsize=8, color=INK2)
save_fig(fig, "fig2_bridges_sido.png")
print("그림 2 저장:", OUT / "fig2_bridges_sido.png")

## 5. 한계와 데이터 완결성

아래 진단은 한계를 수치로 가늠하는 집계다. 개별 항목은 내지 않는다.

- **OSM 완결성**: 공식 설비 통계와 대조하지 않았다. 매핑이 덜 된 곳에서는 변전소에 닿지 않는 끝점, 최대 성분 밖의 작은 조각, 이어지지 않은 선로가 생긴다. 시도별 `lc_share`가 낮은 곳이 그 증상이다.
- **지중 케이블 누락 가능성**: 도심의 154 kV 지중 구간은 OSM에서 덜 그려지기 쉽다. `power=cable` 길이 비율이 낮고 도심 시도의 최대 성분 밖 노드가 많으면 누락을 의심한다. 누락은 브리지 비율을 키우고 연결성을 줄인다.
- **전압 태그 결측**: 전압 태그가 없는 선로는 등급을 알 수 없어 그래프에서 빠진다. 아래 진단은 그중 그래프에 닿는 것의 양이다. `voltage` 오기(354, 380, 145 등)는 등급 구간으로 흡수했고, 다중 값은 최댓값을 쓴다(한 회랑에 345·154 kV가 함께 걸린 경우 하나의 전압으로 본다).
- **회선 수**: `circuits`(없으면 `cables` / 3)를 모르는 구간은 1회선으로 둔다. 구간 단위 브리지는 이중 회선을 한 엣지로 보아 과대, 회선 단위 브리지는 모르는 구간을 1회선으로 보아 상한이다.
- **HVDC(제주)**: `frequency=0` 케이블은 서셉턴스 선로가 아니라 제어 주입으로 다뤄야 하므로(ADR 0012) AC 그래프에서 뺀다. 그러면 제주 AC 계통이 최대 성분 밖의 별도 성분이 된다. "HVDC 포함" 민감도는 구조 연결만 본 것이며 전기적 의미가 없다.
- **대용 리액턴스**: 전압별 x'를 같게 두고 변압기를 무시한다. 절댓값이 아니라 구조 비교용이다. 같은 변전소 노드가 여러 전압 등급을 잇는다.
- **공개성**: 계통 상세는 공개가 제한적이다. 이 결과는 OSM 공개 자료의 추정이며, 실제 계통의 취약성을 말하지 않는다.

In [ ]:
# 5-1. 완결성·한계 진단 (집계만)
diag_unknown = pl.diagnose_unknown_voltage(lines, subs, CFG)
diag_mixed = pl.diagnose_mixed_grade_nodes(lines, subs, CFG)
real = lines.loc[~lines["internal"]]
hv_ways = lines.loc[lines["hvdc"]]
tx = real.loc[real["grade"].isin(["765", "345", "154"]) & real["power"].isin(["line", "cable"])]
n_known, n_kept = DIAG["n_site_clusters_known_kv"], DIAG["n_site_nodes_kept_known_kv"]
LIMITS = {
    "unknown_voltage": diag_unknown,
    "unknown_voltage_share_of_transmission_length": diag_unknown["length_km"] / (tx["length_km"].sum() + diag_unknown["length_km"]),
    "mixed_grade_shared_nodes": diag_mixed,
    "substation_clusters_known_kv": n_known,
    "substation_clusters_known_kv_with_line": n_kept,
    "substation_clusters_known_kv_without_line_share": 1 - n_kept / n_known,
    "cable_length_share_of_transmission": float(tx.loc[tx["power"] == "cable", "length_km"].sum() / tx["length_km"].sum()),
    "underground_tagged_share_of_transmission_length": float(tx.loc[tx["underground"], "length_km"].sum() / tx["length_km"].sum()),
    "circuits_known_share_graph_edges": float(edges1["circuits_n"].notna().mean()),
    "hvdc_ways": int(len(hv_ways)), "hvdc_length_km": float(hv_ways["length_km"].sum()), "hvdc_converter_polygons": int(len(osm["converters"])),
    "lc_share_by_sido": {r.sido_name: round(float(r.lc_share), 3) for r in sido1.itertuples() if r.sido_name != "전국"},
}
print(json.dumps(LIMITS, ensure_ascii=False, indent=1, default=_json_default))

## 6. 요약과 실행 기록

In [ ]:
# 6-1. 요약 JSON과 실행 기록(config_hash, data_snapshot, code_version, seed)
def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as exc:  # 기록용이라 실패해도 실행은 계속한다
        return f"unavailable: {exc}"


def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


SUMMARY = {
    "graph": {k: S[k] for k in ["n_nodes", "n_nodes_substation", "n_nodes_junction", "n_nodes_end", "n_edges", "length_km_total",
                                 "n_components", "comp_sizes_top5", "n_components_size_le3", "lc_nodes", "lc_node_share", "lc_length_share"]},
    "assembly": DIAG,
    "subgraphs": {g: {k: v for k, v in d_.items()} for g, d_ in SUB.items()},
    "metrics": {k: S[k] for k in M_KEYS},
    "leverage_ge_0.9_share": SHARE_LEV90,
    "rbar_circuit_adjusted": RBAR_CIRC,
    "limits": LIMITS,
    "tag_fill": {
        "lines": inv_l["fill"].to_dict("records"),
        "substations": inv_s["by_kind"].to_dict("records"),
        "plants": inv_p["by_kind"].to_dict("records"),
        "generators": inv_g["by_kind"].to_dict("records"),
    },
}
save_json(SUMMARY, "summary.json")
RUN = {
    "config_hash": CONFIG_HASH,
    "config": {"base": CFG, "scenarios": SCENARIOS, "contract": CONTRACT_NAME},
    "data_snapshot": {
        "pbf": {"path": PBF_REL, "md5": PBF_MD5, "bytes": PBF.stat().st_size, "manifest": manifest[PBF_REL]["retrieved_at"], "license": "ODbL 1.0"},
        "regions": {"path": "spike/regions.parquet", "sha256": sha256_of(DATA / "regions.parquet"), "bytes": (DATA / "regions.parquet").stat().st_size},
    },
    "code_version": {
        "git_head": git("rev-parse", "HEAD"),
        "git_dirty_paths": git("status", "--porcelain", "--", "notebooks/spike/power", "docs/decisions").splitlines(),
        "power_lib_sha256": sha256_of(NB_DIR / "power_lib.py"),
        "notebook_sha256": sha256_of(NB_DIR / "05_power_grid.ipynb"),
    },
    "seed": SEED,
    "seed_note": "합성 검산의 무작위 그래프에만 쓴다. 본 계산(그래프 조립·브리지·유효 저항)은 결정적이다",
    "started_at": datetime.datetime.fromtimestamp(T_START).isoformat(timespec="seconds"),
    "elapsed_s": round(time.time() - T_START, 1),
    "outputs": sorted(set(WRITTEN) | {"power_run_meta.json"}),
    "outputs_dir": str(OUT),
}
save_json(RUN, "power_run_meta.json")
print(f"완료 {RUN['elapsed_s']}s. 결과 파일 {len(set(WRITTEN) | {'power_run_meta.json'})}개를 {OUT} 에 썼다.")
print("실행 기록:", {k: RUN[k] for k in ("config_hash", "seed")}, "| git", RUN["code_version"]["git_head"][:10], "| 자료 MD5", PBF_MD5[:10])